In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('deliveries.csv')

In [3]:
df.head()

,match_id,inning,batting_team,bowling_team,over,ball,batter,bowler,non_striker,batsman_runs,extra_runs,total_runs,extras_type,is_wicket,player_dismissed,dismissal_kind,fielder
0,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,SC Ganguly,P Kumar,BB McCullum,0,1,1,legbyes,0,NaN,NaN,NaN
1,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,BB McCullum,P Kumar,SC Ganguly,0,0,0,NaN,0,NaN,NaN,NaN
2,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,BB McCullum,P Kumar,SC Ganguly,0,1,1,wides,0,NaN,NaN,NaN
3,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,4,BB McCullum,P Kumar,SC Ganguly,0,0,0,NaN,0,NaN,NaN,NaN
4,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,5,BB McCullum,P Kumar,SC Ganguly,0,0,0,NaN,0,NaN,NaN,NaN


In [4]:
df.shape

(260920, 17)

In [5]:
df.isnull().sum()

match_id                 0
inning                   0
batting_team             0
bowling_team             0
over                     0
ball                     0
batter                   0
bowler                   0
non_striker              0
batsman_runs             0
extra_runs               0
total_runs               0
extras_type         246795
is_wicket                0
player_dismissed    247970
dismissal_kind      247970
fielder             251566
dtype: int64

## Data Cleaning

In [6]:
# Check for missing values and data types
print("Data Types:")
print(df.dtypes)
print("\n" + "="*50)
print("Missing Values:")
print(df.isnull().sum())
print("\n" + "="*50)
print("Unique values in key columns:")
print(f"Unique match_ids: {df['match_id'].nunique()}")
print(f"Unique batting teams: {df['batting_team'].nunique()}")
print(f"Unique bowling teams: {df['bowling_team'].nunique()}")

Data Types:
match_id            int64
inning              int64
batting_team          str
bowling_team          str
over                int64
ball                int64
batter                str
bowler                str
non_striker           str
batsman_runs        int64
extra_runs          int64
total_runs          int64
extras_type           str
is_wicket           int64
player_dismissed      str
dismissal_kind        str
fielder               str
dtype: object

Missing Values:
match_id                 0
inning                   0
batting_team             0
bowling_team             0
over                     0
ball                     0
batter                   0
bowler                   0
non_striker              0
batsman_runs             0
extra_runs               0
total_runs               0
extras_type         246795
is_wicket                0
player_dismissed    247970
dismissal_kind      247970
fielder             251566
dtype: int64

Unique values in key columns:
Unique match

In [7]:
# Replace 'NA' strings with actual NaN values
df = df.replace('NA', np.nan)
df = df.replace('', np.nan)

print("After replacing 'NA' strings:")
print(df.isnull().sum())

After replacing 'NA' strings:
match_id                 0
inning                   0
batting_team             0
bowling_team             0
over                     0
ball                     0
batter                   0
bowler                   0
non_striker              0
batsman_runs             0
extra_runs               0
total_runs               0
extras_type         246795
is_wicket                0
player_dismissed    247970
dismissal_kind      247970
fielder             251566
dtype: int64


In [8]:
# Convert data types to appropriate formats
df['match_id'] = df['match_id'].astype('int64')
df['inning'] = df['inning'].astype('int64')
df['over'] = df['over'].astype('int64')
df['ball'] = df['ball'].astype('int64')
df['batsman_runs'] = df['batsman_runs'].astype('int64')
df['extra_runs'] = df['extra_runs'].astype('int64')
df['total_runs'] = df['total_runs'].astype('int64')
df['is_wicket'] = df['is_wicket'].astype('int64')

print("Updated Data Types:")
print(df.dtypes)

Updated Data Types:
match_id            int64
inning              int64
batting_team          str
bowling_team          str
over                int64
ball                int64
batter                str
bowler                str
non_striker           str
batsman_runs        int64
extra_runs          int64
total_runs          int64
extras_type           str
is_wicket           int64
player_dismissed      str
dismissal_kind        str
fielder               str
dtype: object


In [9]:
# Check for duplicates
print("Duplicate rows:")
duplicate_count = df.duplicated().sum()
print(f"Total duplicates: {duplicate_count}")

if duplicate_count > 0:
    print("\nRemoving duplicate rows...")
    df = df.drop_duplicates()
    print(f"Dataset shape after removing duplicates: {df.shape}")

Duplicate rows:
Total duplicates: 0


In [10]:
# Data validation - check for logical inconsistencies
print("Data Validation Checks:")
print("="*50)

# Check 1: total_runs = batsman_runs + extra_runs
runs_check = (df['total_runs'] == df['batsman_runs'] + df['extra_runs']).sum()
print(f"Rows where total_runs = batsman_runs + extra_runs: {runs_check}/{len(df)}")

# Check 2: is_wicket should be 0 or 1
wicket_check = df['is_wicket'].isin([0, 1]).sum()
print(f" Rows where is_wicket is 0 or 1: {wicket_check}/{len(df)}")

# Check 3: over and ball ranges
over_max = df['over'].max()
ball_max = df['ball'].max()
print(f"✓ Over range: 0 to {over_max}")
print(f"✓ Ball range: 1 to {ball_max}")

# Check 4: Wicket logic - if is_wicket = 1, player_dismissed should not be NaN
wicket_rows = df[df['is_wicket'] == 1]
wicket_dismissed = wicket_rows['player_dismissed'].notna().sum()
print(f"✓ Wicket rows with player_dismissed: {wicket_dismissed}/{len(wicket_rows)}")

print("="*50)

Data Validation Checks:
✓ Rows where total_runs = batsman_runs + extra_runs: 260920/260920
✓ Rows where is_wicket is 0 or 1: 260920/260920
✓ Over range: 0 to 19
✓ Ball range: 1 to 11
✓ Wicket rows with player_dismissed: 12950/12950


In [11]:
# Handle missing values in specific columns
print("Handling missing values:")
print("="*50)

# For extras_type, dismissal_kind, fielder - fill NaN with 'None' since they're not applicable for non-extra/non-wicket deliveries
df['extras_type'] = df['extras_type'].fillna('None')
df['dismissal_kind'] = df['dismissal_kind'].fillna('None')
df['fielder'] = df['fielder'].fillna('None')

print("Filling categorical columns with 'None':")
print(df[['extras_type', 'dismissal_kind', 'fielder']].isnull().sum())

# Check final missing values
print("\nFinal Missing Values Summary:")
print(df.isnull().sum())
print("="*50)

Handling missing values:
Filling categorical columns with 'None':
extras_type       0
dismissal_kind    0
fielder           0
dtype: int64

Final Missing Values Summary:
match_id                 0
inning                   0
batting_team             0
bowling_team             0
over                     0
ball                     0
batter                   0
bowler                   0
non_striker              0
batsman_runs             0
extra_runs               0
total_runs               0
extras_type              0
is_wicket                0
player_dismissed    247970
dismissal_kind           0
fielder                  0
dtype: int64


In [12]:
# Final cleaned dataset summary
print("CLEANED DATASET SUMMARY")
print("="*50)
print(f"Total Rows: {len(df)}")
print(f"Total Columns: {len(df.columns)}")
print(f"\nDataset Info:")
print(df.info())
print(f"\nFirst few rows of cleaned data:")
print(df.head())
print(f"\nBasic Statistics:")
print(df.describe())
print("="*50)

CLEANED DATASET SUMMARY
Total Rows: 260920
Total Columns: 17

Dataset Info:
<class 'pandas.DataFrame'>
RangeIndex: 260920 entries, 0 to 260919
Data columns (total 17 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   match_id          260920 non-null  int64
 1   inning            260920 non-null  int64
 2   batting_team      260920 non-null  str  
 3   bowling_team      260920 non-null  str  
 4   over              260920 non-null  int64
 5   ball              260920 non-null  int64
 6   batter            260920 non-null  str  
 7   bowler            260920 non-null  str  
 8   non_striker       260920 non-null  str  
 9   batsman_runs      260920 non-null  int64
 10  extra_runs        260920 non-null  int64
 11  total_runs        260920 non-null  int64
 12  extras_type       260920 non-null  str  
 13  is_wicket         260920 non-null  int64
 14  player_dismissed  12950 non-null   str  
 15  dismissal_kind    260920 non-null  st

In [13]:
# Save the cleaned dataset
df.to_csv('deliveries_cleaned.csv', index=False)
print("✓ Cleaned dataset saved as 'deliveries_cleaned.csv'")

✓ Cleaned dataset saved as 'deliveries_cleaned.csv'
